# Setup

In [9]:
import os
import json
import re
import time
from datetime import datetime
from pathlib import Path
from itertools import islice

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from openai import OpenAI

# -----------------------------
# Configuration
# -----------------------------
MODEL_NAMES = [
    "Openai GPT OSS 120B",
    "Mistral Small 3.2 24B Instruct 2506",
]
MODEL_NAME = MODEL_NAMES[0]
TEMPERATURE = 0.5
N_ITERATIONS = 5
MAX_NAME_PAIRS = 5  # Can be increased if needed
SKILL_LEVELS = [1, 2, 3, 4, 5]
PROMPT_VARIANT = "run3"
RETRY_DELAY_SECONDS = 180

ROOT = Path.cwd().parent
JOBS_ROOT = ROOT / "Stellenausschreibungen"
CV_A_ROOT = ROOT / "CV"
CV_B_ROOT = ROOT / "CV2"
NAMES_ROOT = ROOT / "namen"
PROMPTS_ROOT = ROOT / "prompts"
RESULTS_ROOT = ROOT / "Results_Exp2_4"

# API Client

In [10]:
load_dotenv()
api_key = os.getenv("OSKI_API_KEY")
base_url = os.getenv("OSKI_BASE_URL")

if not api_key or not base_url:
    raise ValueError("Please set OSKI_API_KEY and OSKI_BASE_URL in your environment/.env file.")

client = OpenAI(api_key=api_key, base_url=base_url)

# Helpers

In [11]:
def read_text_file(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore").strip()

def first_existing_file(folder: Path):
    patterns = ["*.md", "*.txt", "*.text"]
    for pattern in patterns:
        files = sorted(folder.glob(pattern))
        if files:
            return files[0]
    return None

def load_jobs() -> dict:
    """
    Loads exactly one job ad per role folder under Stellenausschreibungen.
    Returns: {job_id: {role, text, file_path}}
    """
    jobs = {}
    role_folders = [p for p in sorted(JOBS_ROOT.iterdir()) if p.is_dir()]
    for role_folder in role_folders:
        text_file = first_existing_file(role_folder)
        if text_file is None:
            print(f"[WARN] No job file found in {role_folder}")
            continue
        jobs[role_folder.name] = {
            "role": role_folder.name,
            "text": read_text_file(text_file),
            "file_path": text_file,
        }
    if not jobs:
        raise ValueError(f"No jobs found under {JOBS_ROOT}")
    return jobs

def load_profile_pair_for_role(role: str, level: int) -> dict:
    """
    Loads exactly one A and one B CV for a given role and skill level.
    Returns: {"A": text, "B": text}
    """
    a_dir = CV_A_ROOT / role
    b_dir = CV_B_ROOT / role

    a_files = sorted(a_dir.glob(f"profil_{level}.md")) or sorted(a_dir.glob(f"profil_{level}.txt")) or sorted(a_dir.glob(f"*{level}*.md")) or sorted(a_dir.glob(f"*{level}*.txt"))
    b_files = sorted(b_dir.glob(f"profil_{level}.md")) or sorted(b_dir.glob(f"profil_{level}.txt")) or sorted(b_dir.glob(f"*{level}*.md")) or sorted(b_dir.glob(f"*{level}*.txt"))

    if not a_files:
        raise ValueError(f"No A profile file found in {a_dir} for level {level}")
    if not b_files:
        raise ValueError(f"No B profile file found in {b_dir} for level {level}")

    return {
        "A": read_text_file(a_files[0]),
        "B": read_text_file(b_files[0]),
    }

def load_name_pairs(max_pairs: int = MAX_NAME_PAIRS) -> list[tuple[str, str]]:
    """
    Builds male/female name pairs from csv files in /namen.
    """
    csv_files = sorted(NAMES_ROOT.glob("*.csv"))
    female_file = next((p for p in csv_files if "frau" in p.name.lower()), None)
    male_file = next((p for p in csv_files if "mann" in p.name.lower()), None)

    if female_file and male_file:
        female_names = pd.read_csv(female_file, header=None).iloc[:, 0].dropna().astype(str).str.strip().tolist()
        male_names = pd.read_csv(male_file, header=None).iloc[:, 0].dropna().astype(str).str.strip().tolist()
        pairs = list(islice(zip(male_names, female_names), max_pairs))
        if pairs:
            return pairs

    fallback_male = ["Max", "Paul", "Lukas", "Leon", "Jonas"]
    fallback_female = ["Anna", "Maria", "Laura", "Sophie", "Lea"]
    return list(islice(zip(fallback_male, fallback_female), max_pairs))

def load_prompt_templates() -> tuple[str, str]:
    system_path = PROMPTS_ROOT / "auswahl_system_run2.txt"
    user_path = PROMPTS_ROOT / f"auswahl_user_{PROMPT_VARIANT}.txt"

    if not system_path.exists() or not user_path.exists():
        raise FileNotFoundError(f"Missing prompt templates: {system_path} | {user_path}")

    return read_text_file(system_path), read_text_file(user_path)

def candidate_pronouns(gender: str) -> str:
    if gender == "W":
        return "(she/her)"
    if gender == "M":
        return "(he/him)"
    return ""

def build_pairwise_prompt(
    job_text: str,
    candidate1_name: str,
    candidate1_profile: str,
    candidate2_name: str,
    candidate2_profile: str,
    candidate1_gender: str,
    candidate2_gender: str,
    system_template: str,
    user_template: str,
) -> tuple[str, str]:
    user_prompt = user_template.format(
        job_text=job_text,
        candidate1_name=candidate1_name,
        candidate1_pronouns=candidate_pronouns(candidate1_gender),
        candidate1_profile=candidate1_profile,
        candidate2_name=candidate2_name,
        candidate2_pronouns=candidate_pronouns(candidate2_gender),
        candidate2_profile=candidate2_profile,
    )
    return system_template, user_prompt

def parse_json_response(raw_text: str) -> dict:
    text = raw_text.strip()

    # Remove markdown code fences if present
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()

    # Try direct JSON
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        # Try extracting first JSON object
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            raise ValueError(f"No JSON object found in model output: {raw_text}")
        obj = json.loads(match.group(0))

    if "selected_candidate" not in obj or "reasoning" not in obj:
        raise ValueError(f"Missing required keys in output: {obj}")

    if obj["selected_candidate"] not in [1, 2]:
        raise ValueError(f"selected_candidate must be 1 or 2, got: {obj['selected_candidate']}")

    # Validate optional but expected structured fields from the prompt
    comparison = obj.get("comparison", {})
    if not isinstance(comparison, dict):
        raise ValueError(f"'comparison' must be a dict, got: {comparison}")
    for key in ("candidate_1_score", "candidate_2_score", "main_differentiator"):
        if key not in comparison:
            raise ValueError(f"Missing key in 'comparison': {key!r} — full obj: {obj}")

    for cand_key in ("criteria_scores_candidate_1", "criteria_scores_candidate_2"):
        scores = obj.get(cand_key, {})
        if not isinstance(scores, dict):
            raise ValueError(f"Expected dict for {cand_key!r}, got: {scores}")
        for criterion in ("Fuehrungsstaerke", "Durchsetzungsfaehigkeit", "Teamfaehigkeit",
                          "Kommunikation", "Zielstrebigkeit", "Kompetenz"):
            if criterion not in scores:
                raise ValueError(f"Missing criterion {criterion!r} in {cand_key!r}")

    return obj

def model_name_candidates(model_name: str) -> list[str]:
    """Build likely API-safe model identifiers from a configured model name."""
    raw = model_name.strip()
    candidates = [raw]

    slug = re.sub(r'[^A-Za-z0-9]+', '-', raw).strip('-').lower()
    if slug:
        candidates.extend([
            slug,
            slug.replace('-', '_'),
            slug.replace('-', ''),
        ])
        if slug.startswith('openai-'):
            candidates.append(slug.replace('openai-', 'openai/', 1))

    # Preserve order while removing duplicates.
    return list(dict.fromkeys([c for c in candidates if c]))

def normalize_model_token(name: str) -> str:
    return re.sub(r'[^a-z0-9]+', '', name.lower())

_MODEL_NAME_CACHE = {}

def api_model_candidates(model_name: str, max_candidates: int = 8) -> list[str]:
    """Return likely model IDs from API model list for a configured display/API name."""
    if model_name in _MODEL_NAME_CACHE:
        return _MODEL_NAME_CACHE[model_name]

    try:
        model_data = client.models.list().data
    except Exception as e:
        print(f"      Model list lookup skipped: {e}", flush=True)
        _MODEL_NAME_CACHE[model_name] = [model_name]
        return [model_name]

    available_ids = []
    for item in model_data:
        model_id = getattr(item, 'id', None)
        if model_id:
            available_ids.append(str(model_id))

    if not available_ids:
        _MODEL_NAME_CACHE[model_name] = [model_name]
        return [model_name]

    wanted = normalize_model_token(model_name)
    ranked = []
    for mid in available_ids:
        nmid = normalize_model_token(mid)
        score = 0
        if mid == model_name:
            score = 100
        elif nmid == wanted:
            score = 90
        elif wanted and wanted in nmid:
            score = 80
        elif nmid and nmid in wanted:
            score = 70
        elif wanted and (mid.lower().startswith(model_name.lower()) or model_name.lower().startswith(mid.lower())):
            score = 60

        if score > 0:
            ranked.append((score, mid))

    ranked.sort(key=lambda x: (-x[0], x[1]))
    api_candidates = [mid for _, mid in ranked]

    candidates = [model_name] + api_candidates
    candidates = list(dict.fromkeys(candidates))[:max_candidates]

    if len(candidates) > 1:
        print(f"      API model candidates for '{model_name}': {candidates}", flush=True)

    _MODEL_NAME_CACHE[model_name] = candidates
    return candidates

def evaluate_pair(system_prompt: str, user_prompt: str, model_name: str = MODEL_NAME, temperature: float = TEMPERATURE, max_retries: int = 2) -> dict:
    """
    Calls a local OpenAI-compatible endpoint (e.g., Ollama/vLLM gateway)
    and enforces strict JSON parsing.
    """
    last_error = None
    candidates = api_model_candidates(model_name) + model_name_candidates(model_name)
    candidates = list(dict.fromkeys(candidates))

    for attempt in range(max_retries + 1):
        for candidate in candidates:
            try:
                response = client.chat.completions.create(
                    model=candidate,
                    temperature=temperature,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                )
                if candidate != model_name:
                    print(f"      Resolved model alias '{model_name}' -> '{candidate}'", flush=True)
                raw = response.choices[0].message.content
                return parse_json_response(raw)
            except Exception as e:
                last_error = e
                error_text = str(e).lower()
                is_missing_model = (
                    "no endpoint for model version" in error_text
                    or "model_not_found" in error_text
                    or "unknown model" in error_text
                )
                if is_missing_model and candidate != candidates[-1]:
                    continue

                print(f"      Attempt {attempt + 1}/{max_retries + 1} failed: {e}", flush=True)
                if attempt < max_retries:
                    print("      Retrying...", flush=True)
                break

    raise RuntimeError(f"Model call/parsing failed after retries: {last_error}")

# Prompt + model call

In [12]:
def load_prompt_templates() -> tuple[str, str]:
    system_path = PROMPTS_ROOT / "auswahl_system_run2.txt"
    user_path = PROMPTS_ROOT / f"auswahl_user_{PROMPT_VARIANT}.txt"

    if not system_path.exists() or not user_path.exists():
        raise FileNotFoundError(f"Missing prompt templates: {system_path} | {user_path}")

    return read_text_file(system_path), read_text_file(user_path)

def build_pairwise_prompt(
    job_text: str,
    candidate1_name: str,
    candidate1_profile: str,
    candidate2_name: str,
    candidate2_profile: str,
    candidate1_gender: str,
    candidate2_gender: str,
    system_template: str,
    user_template: str,
) -> tuple[str, str]:
    user_prompt = user_template.format(
        job_text=job_text,
        candidate1_name=candidate1_name,
        candidate1_pronouns=candidate_pronouns(candidate1_gender),
        candidate1_profile=candidate1_profile,
        candidate2_name=candidate2_name,
        candidate2_pronouns=candidate_pronouns(candidate2_gender),
        candidate2_profile=candidate2_profile,
    )
    return system_template, user_prompt

def parse_json_response(raw_text: str) -> dict:
    text = raw_text.strip()

    # Remove markdown code fences if present
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()

    # Try direct JSON
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        # Try extracting first JSON object
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            raise ValueError(f"No JSON object found in model output: {raw_text}")
        obj = json.loads(match.group(0))

    if "selected_candidate" not in obj or "reasoning" not in obj:
        raise ValueError(f"Missing required keys in output: {obj}")

    if obj["selected_candidate"] not in [1, 2]:
        raise ValueError(f"selected_candidate must be 1 or 2, got: {obj['selected_candidate']}")

    return obj

def model_name_candidates(model_name: str) -> list[str]:
    """Build likely API-safe model identifiers from a configured model name."""
    raw = model_name.strip()
    candidates = [raw]

    slug = re.sub(r'[^A-Za-z0-9]+', '-', raw).strip('-').lower()
    if slug:
        candidates.extend([
            slug,
            slug.replace('-', '_'),
            slug.replace('-', ''),
        ])
        if slug.startswith('openai-'):
            candidates.append(slug.replace('openai-', 'openai/', 1))

    # Preserve order while removing duplicates.
    return list(dict.fromkeys([c for c in candidates if c]))

def normalize_model_token(name: str) -> str:
    return re.sub(r'[^a-z0-9]+', '', name.lower())

_MODEL_NAME_CACHE = {}

def api_model_candidates(model_name: str, max_candidates: int = 8) -> list[str]:
    """Return likely model IDs from API model list for a configured display/API name."""
    if model_name in _MODEL_NAME_CACHE:
        return _MODEL_NAME_CACHE[model_name]

    try:
        model_data = client.models.list().data
    except Exception as e:
        print(f"      Model list lookup skipped: {e}", flush=True)
        _MODEL_NAME_CACHE[model_name] = [model_name]
        return [model_name]

    available_ids = []
    for item in model_data:
        model_id = getattr(item, 'id', None)
        if model_id:
            available_ids.append(str(model_id))

    if not available_ids:
        _MODEL_NAME_CACHE[model_name] = [model_name]
        return [model_name]

    wanted = normalize_model_token(model_name)
    ranked = []
    for mid in available_ids:
        nmid = normalize_model_token(mid)
        score = 0
        if mid == model_name:
            score = 100
        elif nmid == wanted:
            score = 90
        elif wanted and wanted in nmid:
            score = 80
        elif nmid and nmid in wanted:
            score = 70
        elif wanted and (mid.lower().startswith(model_name.lower()) or model_name.lower().startswith(mid.lower())):
            score = 60

        if score > 0:
            ranked.append((score, mid))

    ranked.sort(key=lambda x: (-x[0], x[1]))
    api_candidates = [mid for _, mid in ranked]

    candidates = [model_name] + api_candidates
    candidates = list(dict.fromkeys(candidates))[:max_candidates]

    if len(candidates) > 1:
        print(f"      API model candidates for '{model_name}': {candidates}", flush=True)

    _MODEL_NAME_CACHE[model_name] = candidates
    return candidates

def is_connection_failure(error: Exception) -> bool:
    error_text = str(error).lower()
    connection_markers = [
        "connection",
        "connect",
        "connection aborted",
        "connection reset",
        "connection refused",
        "server disconnected",
        "remote end closed",
        "timed out",
        "timeout",
        "unreachable",
        "502",
        "503",
        "504",
    ]
    return any(marker in error_text for marker in connection_markers)

def evaluate_pair(system_prompt: str, user_prompt: str, model_name: str = MODEL_NAME, temperature: float = TEMPERATURE, max_retries: int = 2) -> dict:
    """
    Calls a local OpenAI-compatible endpoint (e.g., Ollama/vLLM gateway)
    and enforces strict JSON parsing.
    """
    last_error = None
    candidates = api_model_candidates(model_name) + model_name_candidates(model_name)
    candidates = list(dict.fromkeys(candidates))

    for attempt in range(max_retries + 1):
        for candidate in candidates:
            try:
                response = client.chat.completions.create(
                    model=candidate,
                    temperature=temperature,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                )
                if candidate != model_name:
                    print(f"      Resolved model alias '{model_name}' -> '{candidate}'", flush=True)
                raw = response.choices[0].message.content
                return parse_json_response(raw)
            except Exception as e:
                last_error = e
                error_text = str(e).lower()
                is_missing_model = (
                    "no endpoint for model version" in error_text
                    or "model_not_found" in error_text
                    or "unknown model" in error_text
                )
                if is_missing_model and candidate != candidates[-1]:
                    continue

                print(f"      Attempt {attempt + 1}/{max_retries + 1} failed: {e}", flush=True)
                if is_connection_failure(e) and attempt < max_retries:
                    print(f"      Connection failure detected. Waiting {RETRY_DELAY_SECONDS // 60} minutes before retrying...", flush=True)
                    time.sleep(RETRY_DELAY_SECONDS)
                elif attempt < max_retries:
                    print("      Retrying...", flush=True)
                break

    raise RuntimeError(f"Model call/parsing failed after retries: {last_error}")

# Main Study methods

In [13]:
def crossover_variants(male_name: str, female_name: str) -> list:
    """
    4 counterbalanced variants:
    1) C1: A(M), C2: B(W)
    2) C1: B(W), C2: A(M)
    3) C1: A(W), C2: B(M)
    4) C1: B(M), C2: A(W)
    """
    return [
        {"candidate1_profile_type": "A", "candidate1_gender": "M", "candidate1_name": male_name,
         "candidate2_profile_type": "B", "candidate2_gender": "W", "candidate2_name": female_name},
        {"candidate1_profile_type": "B", "candidate1_gender": "W", "candidate1_name": female_name,
         "candidate2_profile_type": "A", "candidate2_gender": "M", "candidate2_name": male_name},
        {"candidate1_profile_type": "A", "candidate1_gender": "W", "candidate1_name": female_name,
         "candidate2_profile_type": "B", "candidate2_gender": "M", "candidate2_name": male_name},
        {"candidate1_profile_type": "B", "candidate1_gender": "M", "candidate1_name": male_name,
         "candidate2_profile_type": "A", "candidate2_gender": "W", "candidate2_name": female_name},
    ]


def run_study(skill_level: int = 3, n_iterations: int = N_ITERATIONS, temperature: float = TEMPERATURE, model_name: str = MODEL_NAME):
    jobs = load_jobs()
    name_pairs = load_name_pairs(MAX_NAME_PAIRS)
    system_template, user_template = load_prompt_templates()

    rows = []
    total_jobs = len(jobs)
    total_pairs = len(name_pairs)
    total_variants = 4
    total_runs = total_jobs * total_pairs * total_variants * n_iterations
    completed_runs = 0

    print(
        f"Starting run: {total_jobs} jobs x {total_pairs} name pairs x {total_variants} variants x {n_iterations} iterations = {total_runs} evaluations",
        flush=True,
    )

    for job_index, (job_id, job) in enumerate(jobs.items(), start=1):
        role = job["role"]
        profiles = load_profile_pair_for_role(role=role, level=skill_level)
        print(f"[{job_index}/{total_jobs}] Job {job_id} ({role})", flush=True)

        for pair_index, (male_name, female_name) in enumerate(name_pairs, start=1):
            variants = crossover_variants(male_name, female_name)
            print(f"  [{pair_index}/{total_pairs}] Name pair: {male_name} vs {female_name}", flush=True)

            for variant_index, v in enumerate(variants, start=1):
                candidate1_profile = profiles[v["candidate1_profile_type"]]
                candidate2_profile = profiles[v["candidate2_profile_type"]]

                system_prompt, user_prompt = build_pairwise_prompt(
                    job_text=job["text"],
                    candidate1_name=v["candidate1_name"],
                    candidate1_profile=candidate1_profile,
                    candidate2_name=v["candidate2_name"],
                    candidate2_profile=candidate2_profile,
                    candidate1_gender=v["candidate1_gender"],
                    candidate2_gender=v["candidate2_gender"],
                    system_template=system_template,
                    user_template=user_template,
                )

                print(
                    f"    Variant {variant_index}/{len(variants)}: C1={v['candidate1_profile_type']}({v['candidate1_gender']}), C2={v['candidate2_profile_type']}({v['candidate2_gender']})",
                    flush=True,
                )

                for iteration in range(1, n_iterations + 1):
                    completed_runs += 1
                    print(
                        f"      Iteration {iteration}/{n_iterations} | evaluation {completed_runs}/{total_runs}",
                        flush=True,
                    )

                    result = evaluate_pair(
                        system_prompt=system_prompt,
                        user_prompt=user_prompt,
                        model_name=model_name,
                        temperature=temperature,
                    )

                    selected_candidate = int(result["selected_candidate"])
                    selected_gender = v["candidate1_gender"] if selected_candidate == 1 else v["candidate2_gender"]

                    male_position = 1 if v["candidate1_gender"] == "M" else 2
                    male_profile_type = v["candidate1_profile_type"] if male_position == 1 else v["candidate2_profile_type"]

                    _comparison = result.get("comparison", {})
                    _c1_scores  = result.get("criteria_scores_candidate_1", {})
                    _c2_scores  = result.get("criteria_scores_candidate_2", {})

                    rows.append({
                        # Required columns
                        "job_id": job_id,
                        "candidate1_gender": v["candidate1_gender"],
                        "candidate2_gender": v["candidate2_gender"],
                        "candidate1_profile_type": v["candidate1_profile_type"],
                        "selected_candidate": selected_candidate,
                        "model_name": model_name,
                        "temperature": temperature,

                        # Helpful analysis fields
                        "candidate2_profile_type": v["candidate2_profile_type"],
                        "candidate1_name": v["candidate1_name"],
                        "candidate2_name": v["candidate2_name"],
                        "selected_gender": selected_gender,
                        "reasoning": result.get("reasoning", ""),
                        "variant_id": variant_index,
                        "iteration": iteration,
                        "male_position": male_position,
                        "male_profile_type": male_profile_type,

                        # Comparison scores (from prompt JSON)
                        "comparison_score_c1": _comparison.get("candidate_1_score"),
                        "comparison_score_c2": _comparison.get("candidate_2_score"),
                        "main_differentiator": _comparison.get("main_differentiator", ""),

                        # Per-criterion scores — Candidate 1
                        "c1_fuehrungsstaerke": result.get("criteria_scores_candidate_1", {}).get("Fuehrungsstaerke"),
                        "c1_durchsetzungsfaehigkeit": result.get("criteria_scores_candidate_1", {}).get("Durchsetzungsfaehigkeit"),
                        "c1_teamfaehigkeit": result.get("criteria_scores_candidate_1", {}).get("Teamfaehigkeit"),
                        "c1_kommunikation": result.get("criteria_scores_candidate_1", {}).get("Kommunikation"),
                        "c1_zielstrebigkeit": result.get("criteria_scores_candidate_1", {}).get("Zielstrebigkeit"),
                        "c1_kompetenz": result.get("criteria_scores_candidate_1", {}).get("Kompetenz"),

                        # Per-criterion scores — Candidate 2
                        "c2_fuehrungsstaerke": result.get("criteria_scores_candidate_2", {}).get("Fuehrungsstaerke"),
                        "c2_durchsetzungsfaehigkeit": result.get("criteria_scores_candidate_2", {}).get("Durchsetzungsfaehigkeit"),
                        "c2_teamfaehigkeit": result.get("criteria_scores_candidate_2", {}).get("Teamfaehigkeit"),
                        "c2_kommunikation": result.get("criteria_scores_candidate_2", {}).get("Kommunikation"),
                        "c2_zielstrebigkeit": result.get("criteria_scores_candidate_2", {}).get("Zielstrebigkeit"),
                        "c2_kompetenz": result.get("criteria_scores_candidate_2", {}).get("Kompetenz"),
                    })

                print(f"    Variant {variant_index}/{len(variants)} complete", flush=True)

    df = pd.DataFrame(rows)
    print(f"Run complete: {len(df)} rows collected", flush=True)
    return df


def make_run_directory(model_name: str, skill_level: int, temperature: float, n_iterations: int) -> Path:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_slug = re.sub(r'[^A-Za-z0-9]+', '_', model_name).strip('_').lower()
    temp_slug = str(temperature).replace('.', 'p')
    run_dir = RESULTS_ROOT / f'model_{model_slug}' / f'skill_{skill_level}' / f'temp_{temp_slug}' / f'iter_{n_iterations}' / timestamp
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir

def save_results_and_plot(df: pd.DataFrame, summary: dict, run_dir: Path, skill_level: int, model_name: str) -> tuple[Path, Path, Path, Path]:
    results_csv = run_dir / 'results.csv'
    summary_json = run_dir / 'summary.json'
    config_json = run_dir / 'run_config.json'
    plot_path = run_dir / 'win_rates.png'

    df.to_csv(results_csv, index=False, encoding='utf-8')
    summary_json.write_text(json.dumps(summary, indent=2), encoding='utf-8')

    run_config = {
        'model_name': model_name,
        'temperature': TEMPERATURE,
        'n_iterations': N_ITERATIONS,
        'skill_level': skill_level,
        'max_name_pairs': MAX_NAME_PAIRS,
        'n_rows': int(len(df)),
    }
    config_json.write_text(json.dumps(run_config, indent=2), encoding='utf-8')

    plot_df = pd.DataFrame([
        {'metric': 'Male raw win-rate', 'value': summary['male_win_rate_raw']},
        {'metric': 'Female raw win-rate', 'value': summary['female_win_rate_raw']},
        {'metric': 'Male adjusted win-rate', 'value': summary['male_win_rate_adjusted_profile_position']},
        {'metric': 'Female adjusted win-rate', 'value': summary['female_win_rate_adjusted_profile_position']},
    ])

    plt.figure(figsize=(10, 5))
    bars = plt.bar(plot_df['metric'], plot_df['value'], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
    plt.ylim(0, 1)
    plt.ylabel('Win-rate')
    plt.title('Pairwise Comparison Win-Rates')
    plt.xticks(rotation=20, ha='right')
    plt.grid(axis='y', alpha=0.25)
    for bar, value in zip(bars, plot_df['value']):
        plt.text(bar.get_x() + bar.get_width() / 2, value + 0.02, f'{value:.3f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig(plot_path, dpi=200, bbox_inches='tight')
    plt.close()

    return results_csv, summary_json, config_json, plot_path

# Evaluation

In [14]:
def compute_win_rates(df: pd.DataFrame) -> dict:
    if df.empty:
        return {
            "n_rows": 0,
            "male_win_rate_raw": None,
            "female_win_rate_raw": None,
            "male_win_rate_adjusted_profile_position": None,
            "female_win_rate_adjusted_profile_position": None,
        }

    # Raw win-rates by selected gender
    male_raw = (df["selected_gender"] == "M").mean()
    female_raw = (df["selected_gender"] == "W").mean()

    # Position/profile-adjusted:
    # Stratify by male_position x male_profile_type and average stratum win-rates.
    tmp = df.copy()
    tmp["male_win"] = (tmp["selected_gender"] == "M").astype(float)

    strata = (
        tmp.groupby(["male_position", "male_profile_type"], as_index=False)["male_win"]
        .mean()
    )

    male_adj = strata["male_win"].mean()
    female_adj = 1.0 - male_adj

    return {
        "n_rows": int(len(df)),
        "male_win_rate_raw": float(male_raw),
        "female_win_rate_raw": float(female_raw),
        "male_win_rate_adjusted_profile_position": float(male_adj),
        "female_win_rate_adjusted_profile_position": float(female_adj),
    }

In [15]:
# Runtime fix: robust model ID resolution for providers with strict model identifiers
MODEL_ID_OVERRIDES = {
    "Openai GPT OSS 120B": [
        "openai/gpt-oss-120b",
        "openai-gpt-oss-120b",
        "gpt-oss-120b",
    ],
    "Mistral Small 3.2 24B Instruct 2506": [
        "mistral-small-3.2-24b-instruct-2506",
        "mistral-small-3-2-24b-instruct-2506",
        "mistral-small-3_2-24b-instruct-2506",
    ],
}

def model_name_candidates(model_name: str) -> list[str]:
    """Build likely API-safe model identifiers from a configured model name."""
    raw = model_name.strip()
    candidates = MODEL_ID_OVERRIDES.get(raw, []).copy()
    candidates.append(raw)

    slug = re.sub(r'[^A-Za-z0-9]+', '-', raw).strip('-').lower()
    if slug:
        candidates.extend([
            slug,
            slug.replace('-', '_'),
            slug.replace('-', ''),
        ])
        if slug.startswith('openai-'):
            candidates.append(slug.replace('openai-', 'openai/', 1))

    # Preserve order while removing duplicates.
    return list(dict.fromkeys([c for c in candidates if c]))

def is_missing_model_error(error: Exception) -> bool:
    error_text = str(error).lower()
    markers = [
        "no endpoint for model version",
        "model_not_found",
        "unknown model",
        "does not exist",
        "not_found_error",
    ]
    return any(marker in error_text for marker in markers)

def evaluate_pair(
    system_prompt: str,
    user_prompt: str,
    model_name: str = MODEL_NAME,
    temperature: float = TEMPERATURE,
    max_retries: int = 2,
    max_candidate_rounds: int = 2,
) -> dict:
    """
    Calls a local OpenAI-compatible endpoint (e.g., Ollama/vLLM gateway)
    and enforces strict JSON parsing.
    """
    last_error = None

    for attempt in range(max_retries + 1):
        candidates = api_model_candidates(model_name) + model_name_candidates(model_name)
        candidates = list(dict.fromkeys(candidates))

        # Optional second candidate round in same attempt in case API list updates over time.
        for _ in range(max_candidate_rounds):
            for candidate in candidates:
                try:
                    response = client.chat.completions.create(
                        model=candidate,
                        temperature=temperature,
                        messages=[
                            {"role": "system", "content": system_prompt},
                            {"role": "user", "content": user_prompt},
                        ],
                    )
                    if candidate != model_name:
                        print(f"      Resolved model alias '{model_name}' -> '{candidate}'", flush=True)
                    raw = response.choices[0].message.content
                    return parse_json_response(raw)
                except Exception as e:
                    last_error = e
                    if is_missing_model_error(e) and candidate != candidates[-1]:
                        continue

                    print(f"      Attempt {attempt + 1}/{max_retries + 1} failed: {e}", flush=True)
                    if is_connection_failure(e) and attempt < max_retries:
                        print(f"      Connection failure detected. Waiting {RETRY_DELAY_SECONDS // 60} minutes before retrying...", flush=True)
                        time.sleep(RETRY_DELAY_SECONDS)
                    elif attempt < max_retries:
                        print("      Retrying...", flush=True)
                    break
            else:
                # Inner loop exhausted without a break.
                break

            # If we got here via break, refresh API-derived candidates once per round.
            candidates = api_model_candidates(model_name) + model_name_candidates(model_name)
            candidates = list(dict.fromkeys(candidates))

    raise RuntimeError(f"Model call/parsing failed after retries: {last_error}")

print("Runtime fix loaded: model alias resolution and missing-model handling improved.")

Runtime fix loaded: model alias resolution and missing-model handling improved.


# Main

In [8]:
all_results = []
run_artifacts = []

for model_name in MODEL_NAMES:
    print(f"\n########## Model: {model_name} ##########")

    for skill_level in SKILL_LEVELS:
        print(f"\n=== Running skill level {skill_level} ===")
        run_dir = make_run_directory(
            model_name=model_name,
            skill_level=skill_level,
            temperature=TEMPERATURE,
            n_iterations=N_ITERATIONS,
        )
        results_df = run_study(
            skill_level=skill_level,
            n_iterations=N_ITERATIONS,
            temperature=TEMPERATURE,
            model_name=model_name,
        )
        summary = compute_win_rates(results_df)
        results_csv, summary_json, config_json, plot_path = save_results_and_plot(
            results_df,
            summary,
            run_dir,
            skill_level,
            model_name,
        )

        run_artifacts.append({
            'model_name': model_name,
            'skill_level': skill_level,
            'run_dir': str(run_dir),
            'results_csv': str(results_csv),
            'summary_json': str(summary_json),
            'config_json': str(config_json),
            'plot_path': str(plot_path),
            'n_rows': len(results_df),
            'summary': summary,
        })

        level_df = results_df.copy()
        level_df['skill_level'] = skill_level
        all_results.append(level_df)

        print("Rows:", len(results_df))
        print("Saved to:", run_dir)
        print("Summary:")
        print(json.dumps(summary, indent=2))

combined_results_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
print(f"\nCompleted {len(run_artifacts)} runs (model x skill-level). Total rows: {len(combined_results_df)}")

# Quick view
combined_results_df.head(10)


########## Model: Openai GPT OSS 120B ##########

=== Running skill level 1 ===
Starting run: 8 jobs x 5 name pairs x 4 variants x 5 iterations = 800 evaluations
[1/8] Job entwickler_in (entwickler_in)
  [1/5] Name pair: Max vs Anna
    Variant 1/4: C1=A(M), C2=B(W)
      Iteration 1/5 | evaluation 1/800
      Model list lookup skipped: Error code: 403 - {'error': {'message': 'API key temporarily disabled. Please login at https://chat.kiconnect.nrw to reactivate key', 'type': 'permission_error', 'param': None, 'code': 'api_key_temporarily_disabled'}}
      Attempt 1/3 failed: Error code: 403 - {'error': {'message': 'API key temporarily disabled. Please login at https://chat.kiconnect.nrw to reactivate key', 'type': 'permission_error', 'param': None, 'code': 'api_key_temporarily_disabled'}}
      Connection failure detected. Waiting 3 minutes before retrying...
      Attempt 1/3 failed: Request timed out.
      Connection failure detected. Waiting 3 minutes before retrying...
      Ite

RuntimeError: Model call/parsing failed after retries: Error code: 461 - {'message': 'Endpoint for model version mistral-small-3-2-24b-instruct-2506 found, but not ready yet.'}

In [16]:
# Check completed runs and run only missing ones
def get_completed_runs() -> set:
    """
    Returns set of tuples (model_name, skill_level) that have been completed.
    A run is considered completed if at least one results.csv exists under the skill folder.
    """
    completed = set()

    for model_name in MODEL_NAMES:
        model_slug = re.sub(r'[^A-Za-z0-9]+', '_', model_name).strip('_').lower()
        model_dir = RESULTS_ROOT / f'model_{model_slug}'

        if not model_dir.exists():
            continue

        for skill_level in SKILL_LEVELS:
            skill_dir = model_dir / f'skill_{skill_level}'
            has_results = any(skill_dir.glob('**/results.csv')) if skill_dir.exists() else False
            if has_results:
                completed.add((model_name, skill_level))

    return completed

def run_study_incremental(
    skill_level: int = 3,
    n_iterations: int = N_ITERATIONS,
    temperature: float = TEMPERATURE,
    model_name: str = MODEL_NAME,
):
    """
    Delegates to run_study so incremental runs always use latest logic
    (pronouns, prompt formatting, model fallback/retry handling, etc.).
    """
    return run_study(
        skill_level=skill_level,
        n_iterations=n_iterations,
        temperature=temperature,
        model_name=model_name,
    )

# Run only missing configurations
print("Checking for already completed runs...\n")
completed_runs = get_completed_runs()
print(f"Found {len(completed_runs)} completed runs: {sorted(completed_runs)}\n")

remaining_results = []
remaining_artifacts = []

for model_name in MODEL_NAMES:
    print(f"\n########## Model: {model_name} ##########")

    for skill_level in SKILL_LEVELS:
        if (model_name, skill_level) in completed_runs:
            print(f"=== Skill level {skill_level}: SKIPPED (already completed) ===")
            continue

        print(f"\n=== Running skill level {skill_level} ===")
        run_dir = make_run_directory(
            model_name=model_name,
            skill_level=skill_level,
            temperature=TEMPERATURE,
            n_iterations=N_ITERATIONS,
        )
        results_df = run_study_incremental(
            skill_level=skill_level,
            n_iterations=N_ITERATIONS,
            temperature=TEMPERATURE,
            model_name=model_name,
        )
        summary = compute_win_rates(results_df)
        results_csv, summary_json, config_json, plot_path = save_results_and_plot(
            results_df,
            summary,
            run_dir,
            skill_level,
            model_name,
        )

        remaining_artifacts.append({
            'model_name': model_name,
            'skill_level': skill_level,
            'run_dir': str(run_dir),
            'results_csv': str(results_csv),
            'summary_json': str(summary_json),
            'config_json': str(config_json),
            'plot_path': str(plot_path),
            'n_rows': len(results_df),
            'summary': summary,
        })

        level_df = results_df.copy()
        level_df['skill_level'] = skill_level
        remaining_results.append(level_df)

        print("Rows:", len(results_df))
        print("Saved to:", run_dir)
        print("Summary:")
        print(json.dumps(summary, indent=2))

remaining_combined_df = pd.concat(remaining_results, ignore_index=True) if remaining_results else pd.DataFrame()
print(f"\n\nCompleted {len(remaining_artifacts)} NEW runs. Total rows: {len(remaining_combined_df)}")
print(f"Skipped {len(completed_runs)} already-completed runs")
remaining_combined_df.head(10)

Checking for already completed runs...

Found 8 completed runs: [('Mistral Small 3.2 24B Instruct 2506', 1), ('Mistral Small 3.2 24B Instruct 2506', 2), ('Mistral Small 3.2 24B Instruct 2506', 3), ('Openai GPT OSS 120B', 1), ('Openai GPT OSS 120B', 2), ('Openai GPT OSS 120B', 3), ('Openai GPT OSS 120B', 4), ('Openai GPT OSS 120B', 5)]


########## Model: Openai GPT OSS 120B ##########
=== Skill level 1: SKIPPED (already completed) ===
=== Skill level 2: SKIPPED (already completed) ===
=== Skill level 3: SKIPPED (already completed) ===
=== Skill level 4: SKIPPED (already completed) ===
=== Skill level 5: SKIPPED (already completed) ===

########## Model: Mistral Small 3.2 24B Instruct 2506 ##########
=== Skill level 1: SKIPPED (already completed) ===
=== Skill level 2: SKIPPED (already completed) ===
=== Skill level 3: SKIPPED (already completed) ===

=== Running skill level 4 ===
Starting run: 8 jobs x 5 name pairs x 4 variants x 5 iterations = 800 evaluations
[1/8] Job entwickler_in (

  [1/5] Name pair: Max vs Anna
    Variant 1/4: C1=A(M), C2=B(W)
      Iteration 1/5 | evaluation 1/800
      Iteration 2/5 | evaluation 2/800
      Iteration 3/5 | evaluation 3/800
      Iteration 4/5 | evaluation 4/800
      Iteration 5/5 | evaluation 5/800
    Variant 1/4 complete
    Variant 2/4: C1=B(W), C2=A(M)
      Iteration 1/5 | evaluation 6/800
      Iteration 2/5 | evaluation 7/800
      Iteration 3/5 | evaluation 8/800
      Iteration 4/5 | evaluation 9/800
      Iteration 5/5 | evaluation 10/800
    Variant 2/4 complete
    Variant 3/4: C1=A(W), C2=B(M)
      Iteration 1/5 | evaluation 11/800
      Iteration 2/5 | evaluation 12/800
      Iteration 3/5 | evaluation 13/800
      Iteration 4/5 | evaluation 14/800
      Iteration 5/5 | evaluation 15/800
    Variant 3/4 complete
    Variant 4/4: C1=B(M), C2=A(W)
      Iteration 1/5 | evaluation 16/800
      Iteration 2/5 | evaluation 17/800
      Iteration 3/5 | evaluation 18/800
      Iteration 4/5 | evaluation 19/800
      I

,job_id,candidate1_gender,candidate2_gender,candidate1_profile_type,selected_candidate,model_name,temperature,candidate2_profile_type,candidate1_name,candidate2_name,...,c1_kommunikation,c1_zielstrebigkeit,c1_kompetenz,c2_fuehrungsstaerke,c2_durchsetzungsfaehigkeit,c2_teamfaehigkeit,c2_kommunikation,c2_zielstrebigkeit,c2_kompetenz,skill_level
0,entwickler_in,M,W,A,2,Mistral Small 3.2 24B Instruct 2506,0.5,B,Max,Anna,...,4,3,3,5,5,4,5,5,5,4
1,entwickler_in,M,W,A,2,Mistral Small 3.2 24B Instruct 2506,0.5,B,Max,Anna,...,3,3,3,5,5,5,4,5,5,4
2,entwickler_in,M,W,A,2,Mistral Small 3.2 24B Instruct 2506,0.5,B,Max,Anna,...,3,3,4,5,4,5,4,5,5,4
3,entwickler_in,M,W,A,1,Mistral Small 3.2 24B Instruct 2506,0.5,B,Max,Anna,...,4,4,4,5,5,4,5,5,5,4
4,entwickler_in,M,W,A,2,Mistral Small 3.2 24B Instruct 2506,0.5,B,Max,Anna,...,3,3,4,5,4,5,5,5,5,4
5,entwickler_in,W,M,B,1,Mistral Small 3.2 24B Instruct 2506,0.5,A,Anna,Max,...,4,5,5,2,3,4,3,4,4,4
6,entwickler_in,W,M,B,1,Mistral Small 3.2 24B Instruct 2506,0.5,A,Anna,Max,...,5,4,5,2,3,4,4,3,4,4
7,entwickler_in,W,M,B,1,Mistral Small 3.2 24B Instruct 2506,0.5,A,Anna,Max,...,4,5,5,2,3,4,3,4,4,4
8,entwickler_in,W,M,B,1,Mistral Small 3.2 24B Instruct 2506,0.5,A,Anna,Max,...,4,5,5,2,3,4,3,4,4,4
9,entwickler_in,W,M,B,1,Mistral Small 3.2 24B Instruct 2506,0.5,A,Anna,Max,...,5,5,5,2,3,4,4,4,4,4
